# 04 — Customer Engagement Score & Survival Analysis

## Part 1 — Customer Engagement Score
Build a composite engagement score (0–100) from tenure, services, contract type, and payment method, then analyse its relationship with churn.

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

# Load data
X_train = pd.read_csv('../data/X_train.csv')
y_train = pd.read_csv('../data/y_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_test  = pd.read_csv('../data/y_test.csv')

# Combine train features + labels
df_train = pd.concat([X_train, y_train], axis=1)

print(f'Train shape: {df_train.shape}')
print(f'Test  shape: {X_test.shape}')
print(f'Churn rate:  {df_train["Churn"].mean():.1%}')
df_train.head()

In [ ]:
# ── Cell 2: Build Engagement Score (0–100) ────────────────────────────────

def compute_engagement_score(df):
    """Compute a composite engagement score (0–100) for each customer."""

    # --- 1. Tenure score (40%) — normalise to 0-100 (longer = higher) ---
    tenure_min = df['tenure'].min()
    tenure_max = df['tenure'].max()
    tenure_score = (df['tenure'] - tenure_min) / (tenure_max - tenure_min) * 100

    # --- 2. Services score (30%) — count of add-on services used / 6 ---
    service_cols = [
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    services_used = df[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)
    services_score = (services_used / 6) * 100

    # --- 3. Contract score (20%) ---
    contract_map = {
        'Month-to-month': 0,
        'One year': 50,
        'Two year': 100
    }
    contract_score = df['Contract'].map(contract_map)

    # --- 4. Payment score (10%) ---
    payment_map = {
        'Electronic check': 0,
        'Mailed check': 33,
        'Bank transfer (automatic)': 66,
        'Credit card (automatic)': 100
    }
    payment_score = df['PaymentMethod'].map(payment_map)

    # --- Weighted sum ---
    engagement = (
        0.40 * tenure_score +
        0.30 * services_score +
        0.20 * contract_score +
        0.10 * payment_score
    ).round(0).astype(int)

    return engagement


# Apply to train set
df_train['EngagementScore'] = compute_engagement_score(df_train)

print('Engagement Score — descriptive stats:')
print(df_train['EngagementScore'].describe().to_string())
print(f'\nSample scores:')
df_train[['tenure', 'Contract', 'PaymentMethod', 'EngagementScore', 'Churn']].head(10)

In [ ]:
# ── Cell 3: Churn rate by engagement score bin ─────────────────────────────

# Define bins
bins   = [0, 20, 40, 60, 80, 100]
labels = ['0–20', '21–40', '41–60', '61–80', '81–100']

df_train['EngagementBin'] = pd.cut(
    df_train['EngagementScore'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Churn rate per bin
churn_by_bin = (
    df_train
    .groupby('EngagementBin', observed=False)
    .agg(
        Customers=('Churn', 'count'),
        Churned=('Churn', 'sum'),
        ChurnRate=('Churn', 'mean')
    )
)
churn_by_bin['ChurnRate%'] = (churn_by_bin['ChurnRate'] * 100).round(1)

print('Churn Rate by Engagement Score Bin')
print('=' * 55)
print(churn_by_bin[['Customers', 'Churned', 'ChurnRate%']].to_string())
print(f'\nBaseline churn rate: {df_train["Churn"].mean():.1%}')

In [ ]:
# ── Cell 4: Visualize — churn % by engagement score bin ────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']

bars = ax.bar(
    churn_by_bin.index.astype(str),
    churn_by_bin['ChurnRate%'],
    color=colors,
    edgecolor='white',
    linewidth=1.5,
    width=0.65
)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2, height + 0.8,
        f'{height:.1f}%', ha='center', va='bottom',
        fontweight='bold', fontsize=12
    )

# Baseline churn rate line
baseline = df_train['Churn'].mean() * 100
ax.axhline(
    y=baseline, color='#2c3e50', linestyle='--', linewidth=2,
    label=f'Baseline churn rate ({baseline:.1f}%)'
)

ax.set_xlabel('Engagement Score Bin', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Churn Rate by Customer Engagement Score', fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, max(churn_by_bin['ChurnRate%']) * 1.2)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

# Save figure
os.makedirs('../reports/figures', exist_ok=True)
fig.savefig('../reports/figures/churn_by_engagement_score.png', dpi=150, bbox_inches='tight')
print('Saved → reports/figures/churn_by_engagement_score.png')

plt.show()

In [ ]:
# ── Cell 5: Correlation check ─────────────────────────────────────────────

# Pearson correlation between engagement score and churn
corr = df_train['EngagementScore'].corr(df_train['Churn'])
print(f'Correlation (Engagement Score vs Churn): {corr:.4f}')
print()

# Mean engagement score by churn status
mean_scores = (
    df_train
    .groupby('Churn')['EngagementScore']
    .agg(['mean', 'median', 'std'])
    .round(1)
)
mean_scores.index = ['Retained (0)', 'Churned (1)']
print('Mean Engagement Score by Churn Status')
print('=' * 45)
print(mean_scores.to_string())